# 04 · Retrieve — 06 Retraction checking

**Ported from `clinical-search/services/retractions.py`. Nothing in this stage checks whether a top-ranked paper has been retracted -- this closes that gap, and it needs no API key.**

A retracted paper can currently rank first in `05-ranking-and-final-score.ipynb` and nothing downstream would notice. This notebook runs against a small, synthetic retraction list rather than downloading the real ~60 MB Crossref/Retraction Watch dataset the production code points at -- see Step 1 for why, and the stage `README.md` for the caution about never committing that cache.

## What this notebook demonstrates

| Name | What it does | Example |
|---|---|---|
| `load_retractions_from_csv` | Parses a Retraction-Watch-shaped CSV into a set of retracted DOIs | `load_retractions_from_csv(path)` |
| `is_retracted` | Checks one DOI against that set | `is_retracted("10.1000/example-retracted")` |
| `get_retracted_dois` | Cached accessor -- loads once, reused across calls | `get_retracted_dois(cache_path=...)` |


In [ ]:
import sys
from pathlib import Path

_p = Path.cwd().resolve()
for _ in range(6):
    if (_p / "nbio.py").is_file():
        sys.path.insert(0, str(_p))
        break
    _p = _p.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")

import nbio

repo_root = nbio.bootstrap()

## Step 1 — a synthetic retraction list, not the real 60 MB download

The production code downloads Crossref's Retraction Watch dataset
(`api.labs.crossref.org/data/retractionwatch`) to a local CSV cache and
never re-downloads it more often than a 7-day TTL. That's the right design
for a running service; it is the wrong thing for this notebook to do on
every execution -- the cookbook's own exclusions rule is explicit that a
CSV that size must never end up committed to a public repo (it's exactly
what one of the donor repos already had to purge once). So this notebook
ports the *parsing and lookup* logic and demonstrates it against a small,
hand-written CSV shaped like the real one, saved to a temp directory, never
committed.

In [ ]:
import csv
import tempfile
from pathlib import Path

_SYNTHETIC_RETRACTION_CSV = """RetractionDOI,OriginalPaperDOI,RetractionNature
10.9999/retraction-notice-a,10.1000/example-retracted-paper,Retraction
10.9999/retraction-notice-b,10.1000/example-correction-only,Correction
,10.1000/example-retracted-no-notice-doi,Retraction
"""

_tmp_dir = Path(tempfile.mkdtemp(prefix="cookbook-retractions-"))
SYNTHETIC_CSV_PATH = _tmp_dir / "retractions.csv"
SYNTHETIC_CSV_PATH.write_text(_SYNTHETIC_RETRACTION_CSV, encoding="utf-8")
print(f"synthetic retraction CSV written to a temp dir (never committed): {SYNTHETIC_CSV_PATH}")

## Step 2 — `load_retractions_from_csv`

Ported unchanged from `_load_retractions_from_csv`. Only rows whose
`RetractionNature` is exactly `"Retraction"` count -- a `"Correction"` row
(see the synthetic CSV above) is a real Retraction Watch category and must
not be treated as a retraction. Both the retraction notice's own DOI and
the original paper's DOI are added, since a real paper can be cited by
either one depending on which the caller looked up.

In [ ]:
RETRACTION_FILTER = "Retraction"


def load_retractions_from_csv(path: Path) -> set[str]:
    retracted: set[str] = set()
    with path.open(newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row.get("RetractionNature") != RETRACTION_FILTER:
                continue
            rd = (row.get("RetractionDOI") or "").strip()
            od = (row.get("OriginalPaperDOI") or "").strip()
            if rd:
                retracted.add(rd)
            if od:
                retracted.add(od)
    return retracted

## Step 3 — run it and look at real output

Three rows went in; the correction row and the row with no notice DOI
should still each contribute the DOI they do have.

In [ ]:
retracted_dois = load_retractions_from_csv(SYNTHETIC_CSV_PATH)
print(f"{len(retracted_dois)} DOIs marked retracted:")
for doi in sorted(retracted_dois):
    print(" ", doi)

## Step 4 — `is_retracted`, with an in-process cache

Ported from `get_retracted_dois` + `is_retracted`. Production keys the
cache on a 7-day TTL against the downloaded file; this demo skips the TTL
entirely (there is no download to go stale) and just caches the parsed set
on first call, same as the real code does after its own download step.

In [ ]:
_cached_retracted: set[str] | None = None


def get_retracted_dois(cache_path: Path = SYNTHETIC_CSV_PATH) -> set[str]:
    global _cached_retracted
    if _cached_retracted is None:
        _cached_retracted = load_retractions_from_csv(cache_path)
    return _cached_retracted


def is_retracted(doi: str | None) -> bool:
    if not doi or not doi.strip():
        return False
    return doi.strip() in get_retracted_dois()

## Step 5 — check a retracted DOI and a clean one

`10.1000/example-retracted-paper` is in the synthetic list (via its
retraction notice's `OriginalPaperDOI` column); `10.1000/example-clean-paper`
was never in the CSV at all.

In [ ]:
print("retracted paper:", is_retracted("10.1000/example-retracted-paper"))
print("clean paper:     ", is_retracted("10.1000/example-clean-paper"))
print("no DOI at all:   ", is_retracted(None))

assert is_retracted("10.1000/example-retracted-paper") is True
assert is_retracted("10.1000/example-clean-paper") is False
assert is_retracted(None) is False
print()
print("all three checks correct")

## The one real limitation — DOI-only coverage

This check can only screen a candidate that carries a DOI in the first
place. A paper with no DOI on record -- common for older or non-journal
sources -- passes through unscreened, silently, regardless of its actual
retraction status. State that plainly to a reader rather than let the
function's clean `True`/`False` return imply full coverage.

**Where this runs in the pipeline:** ahead of `05-ranking-and-final-score.ipynb`
-- a retracted paper should be dropped or flagged before it's scored and
ranked, not after.